In [1]:
from pathlib import Path
from datetime import datetime, timezone
import itertools
import json

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize_scalar

TASK_DIR = Path.cwd().parent
INPUT_DIR = TASK_DIR / "input"
OUTPUT_DIR = TASK_DIR / "output"

DATA_PATH = INPUT_DIR / "productivity_transitions.csv"
PARAMS_PATH = OUTPUT_DIR / "arw4_params.csv"
SCORES_PATH = OUTPUT_DIR / "arw4_scores.csv"
INITIAL_PATH = OUTPUT_DIR / "arw4_initial_exponential_params.csv"
FIT_PATH = OUTPUT_DIR / "arw4_fit.json"
MANIFEST_PATH = OUTPUT_DIR / "arw4_fit_manifest.json"

MODEL_NAME = "ARW4"
MODEL_TAG = "arw4"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
df = pd.read_csv(DATA_PATH).rename(columns={"CareerAge": "CareerAgeZero","raw_delta": "q_adj_delta"})
data = df[["CareerAgeZero","pubs_adj","q_adj_delta"]].copy()

print(f"Rows: {len(data):,}")
print(f"Transitions: {int(data.CareerAgeZero.min())}-{int(data.CareerAgeZero.max())}")

Rows: 34,109
Transitions: 0-19


In [3]:
def find_mode(vals, bins=50):
    counts, bars = np.histogram(vals,bins=bins)
    i = np.argmax(counts)
    return (bars[i] + bars[i + 1]) / 2

def trunclaplace_negloglike(alpha, xs, ms):
    xs, ms = np.asarray(xs), np.asarray(ms)
    return len(xs) * np.log(alpha) + np.abs(xs - ms).sum() / alpha + np.log(2 - np.exp(-ms / alpha)).sum()

def fit_trunc_laplace(data, ms):
    return minimize_scalar(trunclaplace_negloglike,args=(data,ms),bounds=[0,500],method="Bounded").x

def negloglik_fixed_intercept(beta, scale, xs, ys):
    ks = beta * xs
    if any(ks < 0): return np.inf
    return np.log(2 - np.exp(-ks / scale)).sum() + np.abs(ys - ks).sum() / scale

def mode_regression(xs, ys, prior_mode=None):
    ms = np.repeat(find_mode(xs + ys),len(ys)) if prior_mode is None else prior_mode
    alpha = fit_trunc_laplace(xs + ys,ms)
    result = minimize_scalar(negloglik_fixed_intercept,bounds=(0,5),args=(alpha,xs,xs + ys))
    alpha = fit_trunc_laplace(xs + ys,xs * result.x)
    return alpha, result.x

def last_below_k(arr, k):
    last = 0
    for a in arr:
        if k < a: return last
        last = a
    return last

def first_above_k(arr, k, biggest=20):
    for a in arr:
        if k < a: return a
    return biggest

def aggregate_data_by_career_age(data):
    return {year: np.array([d.pubs_adj,d.q_adj_delta]) for year in range(20) for d in [data[data.CareerAgeZero.eq(year)].sort_values("pubs_adj")]}

def aggregate_within_cutoff(cutoffs, year, career_age_to_data):
    start, end = last_below_k(cutoffs,year), first_above_k(cutoffs,year)
    return np.concatenate([career_age_to_data[y] for y in range(start,end)],axis=1)

def get_all_cutoffs():
    return [c for k in range(1,4) for c in itertools.combinations(range(1,20),k)]

In [4]:
cutoff_set = get_all_cutoffs()
regression_for_cutoffs, regression_scores = {}, []
alpha_global, global_mode = mode_regression(data.pubs_adj.to_numpy(),data.q_adj_delta.to_numpy())
q0 = data.loc[data.CareerAgeZero.eq(0),"pubs_adj"].to_numpy()
alpha_q0 = stats.expon.fit(q0)[1]
nll_q0 = -stats.expon.logpdf(q0,scale=alpha_q0).sum()
career_age_to_data = aggregate_data_by_career_age(data)

for cutoffs in cutoff_set:
    last_cutoff, total_nll, total_nll_fixed, cutoff_data, cutoff_n = 0, nll_q0, nll_q0, [], []
    for cutoff in cutoffs + (np.inf,):
        stage_data = aggregate_within_cutoff(cutoffs,last_cutoff,career_age_to_data)
        x, y = stage_data
        alpha, mode_beta = mode_regression(x,y,prior_mode=global_mode * x)
        mode_mu = find_mode(y)
        nll = trunclaplace_negloglike(alpha,x + y,x * mode_beta)
        nll_fixed = trunclaplace_negloglike(alpha,x + y,x * global_mode)
        cutoff_data.append({"cutoffs": list(cutoffs),"cutoff_start": last_cutoff,"cutoff_end": 20 if np.isinf(cutoff) else int(cutoff),"alpha": alpha,"mode_beta": mode_beta,"mode_mu": mode_mu,"nll_mode": nll,"nll_mode_fixed": nll_fixed,"n": len(x)})
        cutoff_n.append(len(x)); total_nll += nll; total_nll_fixed += nll_fixed; last_cutoff = cutoff
    regression_for_cutoffs[cutoffs] = cutoff_data
    regression_scores.append({"cutoffs": cutoffs,"nll_mode": total_nll,"nll_mode_fixed": total_nll_fixed,"n": cutoff_n,"min_n": min(cutoff_n)})

scores = pd.DataFrame(regression_scores)
scores["num_cutoffs"] = scores.cutoffs.map(len)
scores["k_varying"] = 1 + (scores.num_cutoffs + 1) * 2 + scores.num_cutoffs
scores["k_fixed"] = 1 + (scores.num_cutoffs + 1) + scores.num_cutoffs
scores["aic_varying"] = scores.nll_mode + scores.k_varying
scores["aic_fixed"] = scores.nll_mode_fixed + scores.k_fixed
scores["bic_varying"] = scores.nll_mode + scores.k_varying / 2 * np.log(len(data))
scores["bic_fixed"] = scores.nll_mode_fixed + scores.k_fixed / 2 * np.log(len(data))
selected_cutoffs = scores.sort_values("aic_varying").iloc[0].cutoffs
params = regression_for_cutoffs[selected_cutoffs]

In [5]:
params_df = pd.DataFrame(params)
params_df["stage"] = [f"years_{int(x.cutoff_start) + 1}_{int(x.cutoff_end)}" for x in params_df.itertuples()]
params_df["start"] = params_df.cutoff_start.astype(int) + 1
params_df["end"] = params_df.cutoff_end.astype(int)
params_df = params_df[["stage","start","end","cutoff_start","cutoff_end","n","alpha","mode_beta","mode_mu","nll_mode","nll_mode_fixed"]]

scores_out = scores.copy()
scores_out["cutoffs"] = scores_out.cutoffs.map(str)
scores_out["n"] = scores_out.n.map(str)

fit = {"model": MODEL_NAME,"selected_cutoffs": list(selected_cutoffs),"alpha_q0": alpha_q0,"global_alpha": alpha_global,"global_mode": global_mode,"params": params}
manifest = {"model": MODEL_NAME,"model_tag": MODEL_TAG,"created_utc": datetime.now(timezone.utc).isoformat(),"input": str(DATA_PATH),"rows": len(data),"selected_cutoffs": list(selected_cutoffs)}

params_df.to_csv(PARAMS_PATH,index=False)
scores_out.to_csv(SCORES_PATH,index=False)
pd.DataFrame([{"stage": "year_0","distribution": "exponential","loc": 0.0,"scale_alpha": alpha_q0,"rate_lambda": 1 / alpha_q0,"n": len(q0)}]).to_csv(INITIAL_PATH,index=False)
FIT_PATH.write_text(json.dumps(fit,indent=2))
MANIFEST_PATH.write_text(json.dumps(manifest,indent=2))

print(f"Selected cutoffs: {selected_cutoffs}")
display(params_df)

Selected cutoffs: (1, 5, 11)


,stage,start,end,cutoff_start,cutoff_end,n,alpha,mode_beta,mode_mu,nll_mode,nll_mode_fixed
0,years_1_1,1,1,0,1,2030,3.869801,0.578036,0.602100,5084.677094,5088.077268
1,years_2_5,2,5,1,5,8702,4.615669,0.788798,0.360151,24124.534851,24292.330855
2,years_6_11,6,11,5,11,11885,4.229745,0.716641,-0.307298,32193.393535,32253.584008
3,years_12_20,12,20,11,20,11492,3.608444,0.725396,-0.625005,29225.919005,29320.596555
